In [1]:
# importing libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


Build an automated system that cleans raw supermarket data, detects and removes bad records, creates new analytical features, and stores the final dataset in a SQL database.

# 1.Mount the google drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [3]:
df = pd.read_excel('/content/drive/MyDrive/uk_supermarket_supply_2000 2.xlsx')  # Update the file path as needed
display(df.head(100))

,Date,Region,Store_ID,Category,Opening_Stock,Deliveries,Daily_Sales,Closing_Stock,Lead_Time_Days,Supplier_Status,Stock_Status
0,2025-01-01 00:00:00,Scotland,UK-SCO-79,Meat,162,175,136,201,5,Cancelled,Healthy
1,2025-01-01 00:00:00,Scotland,UK-SCO-39,Fresh Produce,437,94,65,466,7,On-Time,Healthy
2,2025-01-01 00:00:00,Scotland,UK-SCO-90,Meat,360,139,117,382,6,Delayed,Healthy
3,2025-01-01 00:00:00,North West,UK-NOR-34,Meat,396,21,359,58,1,On-Time,Warning
4,2025-01-01 00:00:00,Yorkshire,UK-YOR-13,Meat,85,164,127,122,2,On-Time,Warning
...,...,...,...,...,...,...,...,...,...,...,...
95,17/01/2025,North West,UK-NOR-14,Meat,282,75,200,157,7,Delayed,Healthy
96,18/01/2025,West Midlands,UK-WES-80,Festive,366,19,127,258,6,On-Time,Healthy
97,18/01/2025,North West,UK-NOR-44,Bakery,484,174,607,51,7,On-Time,Warning
98,18/01/2025,West Midlands,UK-WES-52,Dairy,338,141,75,404,7,Delayed,Healthy


# 2.Data Exploration

In [4]:
df.describe()

,Opening_Stock,Deliveries,Daily_Sales,Closing_Stock,Lead_Time_Days
count,2000.000000,2000.00000,2000.000000,2000.000000,2000.000000
mean,281.487500,101.69750,208.442500,185.383500,3.966000
std,131.333477,57.75473,153.492777,135.267594,1.998209
min,50.000000,0.00000,10.000000,0.000000,1.000000
25%,167.000000,52.00000,86.000000,77.000000,2.000000
50%,287.000000,102.00000,178.000000,158.000000,4.000000
75%,394.000000,150.00000,298.000000,271.000000,6.000000
max,500.000000,200.00000,1118.000000,627.000000,7.000000


# 3.Data Standardization

In [5]:
num_cols = ["Opening_Stock", "Deliveries", "Daily_Sales", "Closing_Stock", "Lead_Time_Days"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

In [6]:
df['Date'] = pd.to_datetime(df['Date'])

In [7]:
df.dtypes

,0
Date,datetime64[ns]
Region,object
Store_ID,object
Category,object
Opening_Stock,int64
Deliveries,int64
Daily_Sales,int64
Closing_Stock,int64
Lead_Time_Days,int64
Supplier_Status,object


In [8]:
df.isnull()

,Date,Region,Store_ID,Category,Opening_Stock,Deliveries,Daily_Sales,Closing_Stock,Lead_Time_Days,Supplier_Status,Stock_Status
0,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
1995,False,False,False,False,False,False,False,False,False,False,False
1996,False,False,False,False,False,False,False,False,False,False,False
1997,False,False,False,False,False,False,False,False,False,False,False
1998,False,False,False,False,False,False,False,False,False,False,False


In [9]:
df.isnull().sum()

,0
Date,0
Region,0
Store_ID,0
Category,0
Opening_Stock,0
Deliveries,0
Daily_Sales,0
Closing_Stock,0
Lead_Time_Days,0
Supplier_Status,0


In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df.describe()

,Date,Opening_Stock,Deliveries,Daily_Sales,Closing_Stock,Lead_Time_Days
count,2000,2000.000000,2000.00000,2000.000000,2000.000000,2000.000000
mean,2025-06-30 14:54:57.599999744,281.487500,101.69750,208.442500,185.383500,3.966000
min,2025-01-01 00:00:00,50.000000,0.00000,10.000000,0.000000,1.000000
25%,2025-03-30 00:00:00,167.000000,52.00000,86.000000,77.000000,2.000000
50%,2025-06-30 00:00:00,287.000000,102.00000,178.000000,158.000000,4.000000
75%,2025-10-02 00:00:00,394.000000,150.00000,298.000000,271.000000,6.000000
max,2025-12-31 00:00:00,500.000000,200.00000,1118.000000,627.000000,7.000000
std,NaN,131.333477,57.75473,153.492777,135.267594,1.998209


In [12]:
df['Stock_Status'].unique()

array(['Healthy', 'Warning', 'Critical'], dtype=object)

In [13]:
df['Supplier_Status'].unique()


array(['Cancelled', 'On-Time', 'Delayed'], dtype=object)

In [14]:
df['Daily_Sales'].sum()

np.int64(416885)

# 4.Robust Data Ingestion & Transformation

In [15]:
import pandas as pd

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

In [16]:
 #Standardize datetime
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

# Enforce numeric types
num_cols = ["Opening_Stock", "Deliveries", "Daily_Sales", "Closing_Stock", "Lead_Time_Days"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# Optional: drop broken rows with critical nulls
df = df.dropna(subset=["Date"] + num_cols)

converting all the data columns into proper datetime format and make sure the data

# 5.Heuristic Anomaly Filtering (The "Intelligent Filter")

In [17]:
# Identify anomalies where Daily_Sales is greater than Opening_Stock + Deliveries
anomalies = df[df["Daily_Sales"] > (df["Opening_Stock"] + df["Deliveries"])]

# Save anomalies to a quarantined file for auditing
quarantine_df = anomalies.copy()
quarantine_df.to_csv("quarantined_data.csv", index=False)

# Keep only valid rows in the original dataframe
df = df[df["Daily_Sales"] <= (df["Opening_Stock"] + df["Deliveries"])]

detect analomy in data and identifying impossible sales exceeding stock, separtes them

# 6.Standardization & Status Recalculation

In [18]:
# Create anomaly condition
anomaly_mask = df["Daily_Sales"] > (df["Opening_Stock"] + df["Deliveries"])

# Extract and save anomalous rows
df.loc[anomaly_mask].to_csv("quarantined_data.csv", index=False)

# Keep only valid rows
df = df.loc[~anomaly_mask].copy()

It identifies rows where sales exceed the available stock (opening stock + deliveries), which is mathematically impossible. These invalid records are saved to a separate `quarantined_data.csv` file for auditing purposes. The remaining dataset is then filtered to keep only valid, clean records for further processing.


In [19]:
def recalc_status(closing_stock):
    if closing_stock < 50:
        return "Critical"
    elif closing_stock < 150:
        return "Warning"
    else:
        return "Healthy"

df["Stock_Status"] = df["Closing_Stock"].apply(recalc_status)

In [20]:
print(df[["Closing_Stock", "Stock_Status"]].head(10))

   Closing_Stock Stock_Status
0            201      Healthy
1            466      Healthy
2            382      Healthy
3             58      Warning
4            122      Warning
5            266      Healthy
6             91      Warning
7            107      Warning
8            257      Healthy
9             94      Warning


Recalculates the correct stock status based on closing stock levels using predefined thresholds.
It then overwrites any incorrect labels and displays a sample of the updated results.

# 7.Advanced Programmatic Analysis (Feature Engineering)

In [21]:
def supplier_score(row):
    score = 100

    status = str(row["Supplier_Status"]).lower()

    if status == "on-time":
        score += 10
    elif status == "delayed":
        score -= 30
    elif status == "cancelled":
        score -= 60

    score -= 2 * row["Lead_Time_Days"]

    return max(score, 0)

df["Supplier_Reliability_Score"] = df.apply(supplier_score, axis=1)

In [22]:
print(df[["Supplier_Status", "Lead_Time_Days", "Supplier_Reliability_Score"]].head(10))

  Supplier_Status  Lead_Time_Days  Supplier_Reliability_Score
0       Cancelled               5                          30
1         On-Time               7                          96
2         Delayed               6                          58
3         On-Time               1                         108
4         On-Time               2                         106
5         On-Time               5                         100
6         Delayed               5                          60
7         On-Time               6                          98
8       Cancelled               5                          30
9         On-Time               2                         106


In [23]:
cv_map = df.groupby("Category")["Daily_Sales"].agg(
    lambda x: (x.std() / x.mean()) * 100 if x.mean() != 0 else 0
)

df["Category_Volatility_Index"] = df["Category"].map(cv_map)

In [24]:
print(df[["Category", "Category_Volatility_Index"]].drop_duplicates().head())

         Category  Category_Volatility_Index
0            Meat                  66.232731
1   Fresh Produce                  72.287054
7           Dairy                  72.750203
8          Frozen                  67.787255
10         Bakery                  66.593719


Calculates demand volatility per category using the coefficient of variation.
Maps this volatility score back to each row in the dataset.

# 8.Database Loading

In [25]:
import sqlite3

# Step 5: Database Loading

# Create / connect to the SQLite database file
conn = sqlite3.connect("supermarket.db")

# Load the DataFrame into the SQL table "uk_supermarket_supply"
df.to_sql("uk_supermarket_supply", conn, if_exists="replace", index=False)

# Query to check if data is loaded successfully
result = pd.read_sql("SELECT * FROM uk_supermarket_supply LIMIT 5", conn)

# Print the result
print(result)

# Close the connection
conn.close()

                  Date      Region   Store_ID       Category  Opening_Stock  \
0  2025-01-01 00:00:00    Scotland  UK-SCO-79           Meat            162   
1  2025-01-01 00:00:00    Scotland  UK-SCO-39  Fresh Produce            437   
2  2025-01-01 00:00:00    Scotland  UK-SCO-90           Meat            360   
3  2025-01-01 00:00:00  North West  UK-NOR-34           Meat            396   
4  2025-01-01 00:00:00   Yorkshire  UK-YOR-13           Meat             85   

   Deliveries  Daily_Sales  Closing_Stock  Lead_Time_Days Supplier_Status  \
0         175          136            201               5       Cancelled   
1          94           65            466               7         On-Time   
2         139          117            382               6         Delayed   
3          21          359             58               1         On-Time   
4         164          127            122               2         On-Time   

  Stock_Status  Supplier_Reliability_Score  Category_Volatilit

In [26]:
df.to_csv('uk_supermarket_supply_cleaned.csv', index=False)

In [27]:
from google.colab import files

# Trigger the download of the cleaned dataset
files.download('uk_supermarket_supply_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>